# 19 — Directed Coupling: Is K_nm Asymmetric?

The SCPN assumes symmetric K_nm (K[i,j] = K[j,i]). But if L16 (the strange
loop) introduces directional feedback, then information flow between scales
should be ASYMMETRIC: transfer entropy TE(i→j) ≠ TE(j→i).

The DLA parity result showed: odd (feedback) sector is more robust than
even (projection). This predicts: bottom-up information flow should be
stronger and more reliable than top-down.

**Test**: Compute transfer entropy between all pairs of real physiological
signals from the PhysioNet data. Compare TE matrix to symmetric K_nm.
If TE is significantly asymmetric, the strange loop is measurable.

In [ ]:
import json
import os

import numpy as np
import wfdb
from scipy import signal

In [ ]:
def transfer_entropy(x, y, lag=1, n_bins=10):
    """TE(x→y): information transferred from x to y.

    TE(x→y) = H(y_t | y_{t-1}) - H(y_t | y_{t-1}, x_{t-lag})
    Estimated via binned histograms.
    """
    n = len(x) - lag
    x_past = np.digitize(x[:n], bins=np.linspace(x.min(), x.max(), n_bins)) - 1
    y_past = np.digitize(y[lag - 1 : n + lag - 1], bins=np.linspace(y.min(), y.max(), n_bins)) - 1
    y_fut = np.digitize(y[lag : n + lag], bins=np.linspace(y.min(), y.max(), n_bins)) - 1

    def entropy_cond_1d(a, b):
        """H(A|B) from joint counts."""
        joint = np.zeros((n_bins, n_bins))
        for i in range(len(a)):
            ai, bi = min(a[i], n_bins - 1), min(b[i], n_bins - 1)
            joint[ai, bi] += 1
        joint /= joint.sum() + 1e-12
        pb = joint.sum(axis=0) + 1e-12
        H = 0.0
        for j in range(n_bins):
            if pb[j] > 1e-10:
                p_a_given_b = joint[:, j] / pb[j]
                p_a_given_b = p_a_given_b[p_a_given_b > 1e-12]
                H -= pb[j] * np.sum(p_a_given_b * np.log2(p_a_given_b))
        return H

    def entropy_cond_2d(a, b, c):
        """H(A|B,C) approximated as H(A|B+C*n_bins)."""
        bc = np.clip(b, 0, n_bins - 1) * n_bins + np.clip(c, 0, n_bins - 1)
        n_bc = n_bins * n_bins
        joint = np.zeros((n_bins, n_bc))
        for i in range(len(a)):
            ai = min(a[i], n_bins - 1)
            bci = min(int(bc[i]), n_bc - 1)
            joint[ai, bci] += 1
        joint /= joint.sum() + 1e-12
        pbc = joint.sum(axis=0) + 1e-12
        H = 0.0
        for j in range(n_bc):
            if pbc[j] > 1e-10:
                p_a_given_bc = joint[:, j] / pbc[j]
                p_a_given_bc = p_a_given_bc[p_a_given_bc > 1e-12]
                H -= pbc[j] * np.sum(p_a_given_bc * np.log2(p_a_given_bc))
        return H

    H_y_given_ypast = entropy_cond_1d(y_fut, y_past)
    H_y_given_ypast_xpast = entropy_cond_2d(y_fut, y_past, x_past)

    return max(0.0, H_y_given_ypast - H_y_given_ypast_xpast)


print("Transfer entropy function defined.")

In [ ]:
# Load PhysioNet data
DATA_DIR = (
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/00_DATASETS/physionet_slpdb"
)


def bandpass(data, lo, hi, fs, order=4):
    sos = signal.butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, data)


records = ["slp01a", "slp01b", "slp02a", "slp02b"]

all_asymmetry = []

for rec_name in records:
    record = wfdb.rdrecord(os.path.join(DATA_DIR, rec_name))
    fs = record.fs
    channels = record.sig_name
    data = record.p_signal

    eeg_idx = next((i for i, c in enumerate(channels) if "EEG" in c), None)
    ecg_idx = next((i for i, c in enumerate(channels) if "ECG" in c), None)
    resp_idx = next(
        (i for i, c in enumerate(channels) if "Resp" in c.replace("resp", "Resp")), None
    )
    bp_idx = next((i for i, c in enumerate(channels) if "BP" in c), None)

    if eeg_idx is None or ecg_idx is None:
        continue

    # 5-minute segment, downsampled to 50 Hz for speed
    seg_len = int(5 * 60 * fs)
    start = data.shape[0] // 2 - seg_len // 2

    eeg = data[start : start + seg_len, eeg_idx]
    ecg = data[start : start + seg_len, ecg_idx]
    resp = data[start : start + seg_len, resp_idx] if resp_idx is not None else None
    bp = data[start : start + seg_len, bp_idx] if bp_idx is not None else None

    # Extract bands
    signals = {
        "delta": bandpass(eeg, 0.5, 4.0, fs),
        "theta": bandpass(eeg, 4.0, 8.0, fs),
        "alpha": bandpass(eeg, 8.0, 13.0, fs),
        "beta": bandpass(eeg, 13.0, 30.0, fs),
        "ecg": bandpass(ecg, 0.5, 2.0, fs),
    }
    if resp is not None:
        signals["resp"] = bandpass(resp, 0.1, 0.5, fs)
    if bp is not None:
        signals["bp"] = bandpass(bp, 0.04, 0.15, fs)

    # Downsample to 50 Hz
    ds = fs // 50
    for k in signals:
        signals[k] = signals[k][::ds]

    sig_names = list(signals.keys())
    N = len(sig_names)

    # Compute TE matrix
    TE = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i != j:
                TE[i, j] = transfer_entropy(signals[sig_names[i]], signals[sig_names[j]], lag=5)

    # Asymmetry: |TE(i→j) - TE(j→i)| / (TE(i→j) + TE(j→i))
    asymmetry_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            denom = TE[i, j] + TE[j, i]
            if denom > 1e-10:
                asymmetry_matrix[i, j] = abs(TE[i, j] - TE[j, i]) / denom

    mean_asym = np.mean(asymmetry_matrix[np.triu_indices(N, k=1)])

    print(f"\n=== {rec_name} ===")
    print(f"TE matrix ({N} signals):")
    print(f"{'':>8}", end="")
    for s in sig_names:
        print(f"{s[:6]:>8}", end="")
    print()
    for i, s in enumerate(sig_names):
        print(f"{s[:8]:>8}", end="")
        for j in range(N):
            print(f"{TE[i, j]:8.4f}", end="")
        print()

    # Dominant direction for each pair
    print("\nDirected pairs (TE asymmetry > 20%):")
    for i in range(N):
        for j in range(i + 1, N):
            denom = TE[i, j] + TE[j, i]
            if denom > 0.01:
                asym = (TE[i, j] - TE[j, i]) / denom
                if abs(asym) > 0.2:
                    direction = (
                        f"{sig_names[i]}→{sig_names[j]}"
                        if asym > 0
                        else f"{sig_names[j]}→{sig_names[i]}"
                    )
                    print(f"  {direction:20s} asymmetry={abs(asym):.2f}")

    print(f"Mean asymmetry: {mean_asym:.4f}")
    all_asymmetry.append(
        {"record": rec_name, "mean_asymmetry": round(mean_asym, 4), "n_signals": N}
    )

In [ ]:
# Grand summary
print("\n" + "=" * 60)
print("  GRAND SUMMARY: Is cross-scale coupling DIRECTED?")
print("=" * 60)
mean_all = np.mean([r["mean_asymmetry"] for r in all_asymmetry])
for r in all_asymmetry:
    print(f"  {r['record']}: mean asymmetry = {r['mean_asymmetry']:.4f}")
print(f"\n  Grand mean asymmetry: {mean_all:.4f}")
print("  (0 = perfectly symmetric, 1 = completely one-directional)")
print()
if mean_all > 0.3:
    print("  SIGNIFICANT ASYMMETRY in cross-scale information flow.")
    print("  K_nm should be a DIRECTED coupling matrix, not symmetric.")
    print("  The strange loop (L16) creates measurable directional bias.")
elif mean_all > 0.15:
    print("  MODERATE ASYMMETRY. Some pairs are directional, others are not.")
    print("  K_nm may need partial asymmetry (directed for some scale pairs).")
else:
    print("  LOW ASYMMETRY. Cross-scale coupling is approximately symmetric.")
    print("  The symmetric K_nm assumption holds for these signals.")

print("\n--- JSON RESULTS ---")
print(
    json.dumps(
        {
            "records": all_asymmetry,
            "grand_mean_asymmetry": round(mean_all, 4),
        },
        indent=2,
    )
)